# 5 · Bonus lab: a simple LangChain tool-using agent | 30–45 additional minutes
**Optional, after the three-hour core workshop.** Understand an LLM, two tools, and the agent's tool-selection loop. Reuse the RAG pipeline you already built.

Unlike simple RAG, an **agent chooses which tool to use** for each question:
- `support_ticket_metrics` returns **actual SQL aggregates** from your team table.
- `support_kb_search` retrieves **documents with pgvector** (and OpenAI embeddings).

The LLM cannot send arbitrary SQL or execute Python code. The SQL tool only supports three predefined, read-only queries. This protects your shared Supabase instance.

Suggested test prompts:
- "How many tickets do we have by type?"
- "How do I troubleshoot Bluetooth pairing?"
- "Which issue type is most common, and what troubleshooting guidance is available for it?"

In [ ]:
%pip install -q "langchain>=1,<2" "langchain-openai>=1,<2" "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "openai>=1.55,<3"

In [ ]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

In [ ]:
import os, getpass
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass.getpass("Your budgeted OpenAI API key (never instructor master key): ")
client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"      # 1536 dimensions by default
CHAT_MODEL = "gpt-4.1-mini"

def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

def vector_literal(vector):
    assert len(vector) == 1536, "Embedding dimension must match vector(1536)"
    return "[" + ",".join(str(float(x)) for x in vector) + "]"

In [ ]:
def retrieve_articles(question, k=3):
    q_vec = vector_literal(embed(question))
    # Instructor-provisioned views/tables; schema is constructed only from a numeric team ID.
    query = f"""
      SELECT article_id, title, content, 1 - (embedding <=> %s::extensions.vector) AS similarity
      FROM (
         SELECT article_id, title, content, embedding FROM shared.knowledge_articles
         UNION ALL
         SELECT article_id, title, content, embedding FROM {TEAM_SCHEMA}.knowledge_articles
      ) kb
      WHERE embedding IS NOT NULL
      ORDER BY embedding <=> %s::extensions.vector
      LIMIT %s
    """
    return fetch_df(query, (q_vec,q_vec,int(k)))

def grounded_answer(question, passages):
    context = "\n\n".join(
        f"[{row.article_id}] {row.title}: {row.content}" for row in passages.itertuples()
    )
    response = client.responses.create(
        model=CHAT_MODEL,
        input=[
          {"role":"system","content": "Answer only from retrieved knowledge articles. Cite the article IDs in square brackets. If context does not answer the question, say you do not know. Do not invent policies."},
          {"role":"user","content":f"Question: {question}\n\nRetrieved articles:\n{context}"},
        ],
    )
    return response.output_text

## A. See the two real tools

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def support_kb_search(question: str) -> str:
    """Search official workshop support KB articles for troubleshooting or policy guidance; return article IDs."""
    hits = retrieve_articles(question, k=3)
    return "\n".join(
        f"[{r.article_id}] {r.title}: {r.content}" for r in hits.itertuples()
    ) or "No relevant articles found"

@tool
def support_ticket_metrics(metric: str) -> str:
    """Get structured ticket analytics. metric must be: 'summary', 'by_type', or 'by_channel'. No arbitrary SQL."""
    choices = {
        "summary": f"""SELECT COUNT(*) AS tickets,
            COUNT(*) FILTER (WHERE status='Closed') AS closed,
            ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction
            FROM {TEAM_SCHEMA}.clean_tickets""",
        "by_type": f"""SELECT ticket_type, COUNT(*) AS tickets
            FROM {TEAM_SCHEMA}.clean_tickets GROUP BY ticket_type ORDER BY tickets DESC""",
        "by_channel": f"""SELECT channel, COUNT(*) AS tickets
            FROM {TEAM_SCHEMA}.clean_tickets GROUP BY channel ORDER BY tickets DESC""",
    }
    if metric not in choices:
        return "Unsupported metric; choose summary, by_type, or by_channel."
    return fetch_df(choices[metric]).to_json(orient="records")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=OPENAI_API_KEY)
agent = create_agent(
    model=model,
    tools=[support_kb_search, support_ticket_metrics],
    system_prompt=(
        "You are a support analytics agent. Use support_ticket_metrics for numeric/count questions "
        "and support_kb_search for troubleshooting/policy questions. Cite KB article IDs when used. "
        "Do not invent policies or data. You cannot execute arbitrary SQL or mutate data. "
        "If data is not available, explain the limitation."
    ),
)
print("Agent ready: 2 tools registered")

## TODO 1 · Ask the agent to pick a tool

In [ ]:
# TODO 1: Ask a metric question (not a knowledge-base question)
prompt = "TODO: replace with a question about tickets by type"
result = agent.invoke({"messages":[{"role":"user","content":prompt}]})
print("Agent answer:",result["messages"][-1].content)

## B. Explore decisions: what was called?

In [ ]:
for msg in result["messages"]:
    role=type(msg).__name__
    if getattr(msg,"tool_calls",None):
        print(role,"requested tools:",[t["name"] for t in msg.tool_calls])
    elif role=="ToolMessage":
        print("Tool returned:", str(msg.content)[:450])
print("Now try a troubleshooting question and observe a different tool.")

## TODO 2 · Ask a question that requires BOTH tools

In [ ]:
# TODO 2: Write a question that requires ticket counts AND an article-based solution.
both_prompt="TODO: replace with a mixed analytics + troubleshooting question"
result2=agent.invoke({"messages":[{"role":"user","content":both_prompt}]})
print(result2["messages"][-1].content)

## C. Bonus discussion
- Why do **predefined queries** make an SQL agent safer than arbitrary SQL generation?
- Did the model actually use both tools? Inspect tool calls instead of assuming.
- What can go wrong if retrieved articles are not accurate?
- When is a deterministic pipeline simpler than an agent?

**Not production-ready:** no access-control/auth workflow, approval gates, output validation or comprehensive rate limits. This is a workshop demonstration.